In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

Create Spark Session

In [0]:
spark = SparkSession.builder \
    .appName("School Data Transformation Pipeline") \
    .getOrCreate()

Load CSV


In [0]:
df = spark.read.csv("/Volumes/trainning/task-1/tasks/school_performance.csv",header=True,inferSchema=True)

Display data

In [0]:
print("Schema of dataset")
df.printSchema()

print("Total rows:", df.count())

print("Sample data")
df.show(5)

Schema of dataset
root
 |-- record_id: integer (nullable = true)
 |-- school_id: string (nullable = true)
 |-- school_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- grade_level: string (nullable = true)
 |-- teacher_id: string (nullable = true)
 |-- teacher_name: string (nullable = true)
 |-- num_students: integer (nullable = true)
 |-- avg_score: double (nullable = true)
 |-- pass_rate: double (nullable = true)
 |-- exam_date: date (nullable = true)

Total rows: 300
Sample data
+---------+---------+----------------+------+--------+-----------+----------+------------+------------+---------+---------+----------+
|record_id|school_id|     school_name|region| subject|grade_level|teacher_id|teacher_name|num_students|avg_score|pass_rate| exam_date|
+---------+---------+----------------+------+--------+-----------+----------+------------+------------+---------+---------+----------+
|     1001|   SCH004| Hilltop College|  West|

Trim WhiteSpace

In [0]:
df = df.withColumn("teacher_name", trim(col("teacher_name"))) \
       .withColumn("school_name", trim(col("school_name")))

Replace NULL avg_score with subject mean


In [0]:
subject_mean = df.groupBy("subject") \
    .agg(avg("avg_score").alias("subject_mean"))

df = df.join(subject_mean, "subject", "left")

df = df.withColumn(
    "avg_score",
    when(col("avg_score").isNull(), col("subject_mean"))
    .otherwise(col("avg_score"))
).drop("subject_mean")


Add Performance Band Column

In [0]:
df = df.withColumn(
    "performance_band",
    when(col("avg_score") >= 80, "High")
    .when(col("avg_score") >= 60, "Medium")
    .otherwise("Low")
)


Add performance_band

In [0]:
df = df.withColumn(
"performance_band",
F.when(F.col("avg_score") >= 80, "High")
.when(F.col("avg_score") >= 60, "Medium")
.otherwise("Low")
)

Standardize school_name to UPPERCASE


In [0]:
df = df.withColumn("school_name", upper(col("school_name")))

Cast Data Types

In [0]:
df = df.withColumn("avg_score", col("avg_score").cast(DoubleType())) \
       .withColumn("pass_rate", col("pass_rate").cast(DoubleType()))

Aggregations

In [0]:
print("Average score and pass rate by region")

region_stats = df.groupBy("region") \
    .agg(
        avg("avg_score").alias("avg_score"),
        avg("pass_rate").alias("avg_pass_rate")
    )

region_stats.show()

print("Top 3 teachers by average score")

top_teachers = df.groupBy("teacher_name") \
    .agg(avg("avg_score").alias("teacher_avg_score")) \
    .orderBy(desc("teacher_avg_score")) \
    .limit(3)

top_teachers.show()
print("Total students examined per subject")

students_per_subject = df.groupBy("subject") \
    .agg(count("*").alias("total_students"))

students_per_subject.show()

students_per_subject.show()

Average score and pass rate by region
+------+-----------------+------------------+
|region|        avg_score|     avg_pass_rate|
+------+-----------------+------------------+
| North|75.90675675675672|0.8206756756756757|
|  East|78.52564102564104|0.8116666666666669|
| South|74.35342465753425|0.7973972602739725|
|  West|75.12266666666665|0.8134666666666666|
+------+-----------------+------------------+

Top 3 teachers by average score
+------------+-----------------+
|teacher_name|teacher_avg_score|
+------------+-----------------+
|  Dr. Hassan|           78.306|
|   Ms. Patel|77.60943396226415|
|   Ms. Adams|76.21081081081081|
+------------+-----------------+

Total students examined per subject
+-----------+--------------+
|    subject|total_students|
+-----------+--------------+
|    English|            64|
|Mathematics|            59|
|   Computer|            57|
|    History|            45|
|    Science|            75|
+-----------+--------------+

+-----------+--------------+
| 

Save Output


In [0]:
df.write.mode("overwrite").parquet("/mnt/outputs/school_performance_clean.csv")

Data Summary

In [0]:
print("Total rows processed:", df.count())

print("Null count per column")

null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.show()

Total rows processed: 300
Null count per column
+-------+---------+---------+-----------+------+-----------+----------+------------+------------+---------+---------+---------+----------------+
|subject|record_id|school_id|school_name|region|grade_level|teacher_id|teacher_name|num_students|avg_score|pass_rate|exam_date|performance_band|
+-------+---------+---------+-----------+------+-----------+----------+------------+------------+---------+---------+---------+----------------+
|      0|        0|        0|          0|     0|          0|         0|           0|           0|        0|        0|        0|               0|
+-------+---------+---------+-----------+------+-----------+----------+------------+------------+---------+---------+---------+----------------+

